In [ ]:
%pip install polars scikit-learn mlflow numpy matplotlib seaborn sentence-transformers

In [ ]:
import polars as pl
from sklearn.model_selection import cross_validate, train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, accuracy_score, make_scorer
from sklearn.neighbors import KNeighborsClassifier
import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsOneClassifier
import torch, gc
from sklearn.preprocessing import FunctionTransformer
from sklearn.naive_bayes import GaussianNB

In [ ]:
df_reviews_olist = pl.read_csv(
    '../data/olist_order_reviews_dataset.csv'
)
df_reviews_olist

In [ ]:
df_reviews_olist.describe()

In [ ]:
df_reviews_olist = (
    df_reviews_olist
    .select(['review_comment_message', 'review_score']) 
    .drop_nulls(subset=['review_comment_message', 'review_score'])                                         
    .with_columns(                                                                                
        pl.when(pl.col('review_score') <= 2).then(0)
        .when(pl.col('review_score') == 3).then(1)
        .otherwise(2)
        .alias('label')
    )
)

In [ ]:
print(f'Total de instâncias:{df_reviews_olist.shape[0]}')

print(f'\nDistribuição de classes:\n{df_reviews_olist['label'].value_counts(normalize=True) * 100}')

In [ ]:
X = df_reviews_olist['review_comment_message']
y = df_reviews_olist['label']

In [ ]:
vec = TfidfVectorizer()
vec.fit(X)
vocab_size = len(vec.vocabulary_)
print(f"Total de palavras distintas no corpus: {vocab_size}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(
    data=df_reviews_olist,
    x='review_score',
    hue='review_score',
    palette='RdYlGn',
    ax=axes[0],
)
axes[0].set_title("Distribuição por Nota de Avaliação (1, 2, 3, 4, 5)")
axes[0].set_ylabel("Quantidade de Avaliações")

sns.countplot(
    data=df_reviews_olist,
    x='label',
    hue='label',
    palette=["#e74c3c", "#808080", "#2ecc71"],
    ax=axes[1],
)
axes[1].set_title("Distribuição do Vetor Label")
axes[1].set_xticks([0, 1, 2])
axes[1].set_xticklabels(['0 (Ruim)', '1 (Neutro)', '2 (Bom)'])
axes[1].set_ylabel("Quantidade de Avaliações")

plt.tight_layout()
plt.show()


In [ ]:
def to_dense(X):
    return X.toarray()

models = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "SVM_OvO": OneVsOneClassifier(LinearSVC(class_weight='balanced', max_iter=5000, random_state=42)),
    "Decision_Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "GaussianNB": GaussianNB()
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Baseline_Olist_TFIDF")

for model_name, classifier in models.items():

    with mlflow.start_run(run_name=f"{model_name}_Olist"):
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        mlflow.log_param("dataset", "Olist")
        mlflow.log_param("representation", "TF-IDF")
        mlflow.log_param("classifier", model_name)
        mlflow.log_param("max_features", "None")

        if model_name == "GaussianNB":
            pipeline = Pipeline([
                ('tfidf', TfidfVectorizer()),
                ('to_dense', FunctionTransformer(to_dense)),
                ('clf', classifier)
            ])
            n_jobs = 1
        else:
            pipeline = Pipeline([
                ('tfidf', TfidfVectorizer()),
                ('clf', classifier)
            ])
            n_jobs = -1
            
        scoring_metrics = {
            'accuracy': 'accuracy',
            'precision_macro': make_scorer(precision_score, average='macro', zero_division=0),
            'recall_macro': make_scorer(recall_score, average='macro', zero_division=0),
            'f1_macro': make_scorer(f1_score, average='macro', zero_division=0)
        }

        cv_results = cross_validate(
            pipeline, X_train, y_train, 
            cv=skf, 
            scoring=scoring_metrics, 
            n_jobs=n_jobs,
            return_estimator=True
        )
        
        mlflow.log_metric("cv_accuracy_mean", np.mean(cv_results['test_accuracy']))
        mlflow.log_metric("cv_precision_mean", np.mean(cv_results['test_precision_macro']))
        mlflow.log_metric("cv_recall_mean", np.mean(cv_results['test_recall_macro']))
        mlflow.log_metric("cv_f1_macro_mean", np.mean(cv_results['test_f1_macro']))
        
        mlflow.log_metric("cv_f1_macro_std", np.std(cv_results['test_f1_macro']))
        mlflow.log_metric("cv_fit_time_mean", np.mean(cv_results['fit_time']))

        best_fold_index = np.argmax(cv_results['test_f1_macro'])
        best_model = cv_results['estimator'][best_fold_index]
        
        mlflow.log_metric("best_fold_index", best_fold_index)
        
        vocab_size_pipeline = len(best_model.named_steps['tfidf'].vocabulary_)
        mlflow.log_metric("actual_num_features", vocab_size_pipeline)

        y_pred = best_model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_precision", prec)
        mlflow.log_metric("test_recall", rec)
        mlflow.log_metric("test_f1_score", f1)

        if model_name == "GaussianNB":
            mlflow.sklearn.log_model(sk_model=best_model, name="model")
        else:
            mlflow.sklearn.log_model(
            sk_model=best_model,
            name="model",
            serialization_format="skops",
            skops_trusted_types=["scipy.sparse._csr.csr_matrix"]
        )

In [ ]:
X = df_reviews_olist['review_comment_message'].to_list()
y = df_reviews_olist['label'].to_numpy()

In [ ]:
embedding_models = {
    "MiniLM-L6": "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    "DistilBERT-Multi": "sentence-transformers/multi-qa-distilbert-cos-v1",
    "BERTimbau": "neuralmind/bert-base-portuguese-cased",
    "RexBERT-Ecom": "thebajajra/RexBERT-large",
    "BGE-M3": "BAAI/bge-m3",
    "Qwen-GTE": "Alibaba-NLP/gte-Qwen2-1.5B-instruct"
}

models = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "SVM_OvO": OneVsOneClassifier(LinearSVC(class_weight='balanced', max_iter=5000, random_state=42)),
    "Decision_Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "GaussianNB": GaussianNB()
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Embeddings_Olist")

for emb_name, emb_path in embedding_models.items():
    print(f"\n--- Extraindo embeddings usando: {emb_name} ---")
    
    encoder = SentenceTransformer(emb_path, device='cuda', trust_remote_code=True)
    encoder.max_seq_length = 256
    X_embeddings = encoder.encode(X, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    
    for clf_name, classifier in models.items():
        print(f"Treinando classificador: {clf_name}...")

        run_name = f"{emb_name}_{clf_name}" 
        
        with mlflow.start_run(run_name=run_name):
            
            X_train, X_test, y_train, y_test = train_test_split(X_embeddings, y, test_size=0.2, random_state=42, stratify=y)
            
            mlflow.log_param("dataset", "Olist")
            mlflow.log_param("representation", emb_name)
            mlflow.log_param("classifier", clf_name)
            
            scoring_metrics = {
                'accuracy': 'accuracy',
                'precision_macro': make_scorer(precision_score, average='macro', zero_division=0),
                'recall_macro': make_scorer(recall_score, average='macro', zero_division=0),
                'f1_macro': make_scorer(f1_score, average='macro', zero_division=0)
            }

            cv_results = cross_validate(
                classifier, X_train, y_train, 
                cv=skf, scoring=scoring_metrics, n_jobs=-1, return_estimator=True
            )
            
            mlflow.log_metric("cv_accuracy_mean", np.mean(cv_results['test_accuracy']))
            mlflow.log_metric("cv_precision_mean", np.mean(cv_results['test_precision_macro']))
            mlflow.log_metric("cv_recall_mean", np.mean(cv_results['test_recall_macro']))
            mlflow.log_metric("cv_f1_macro_mean", np.mean(cv_results['test_f1_macro']))
            mlflow.log_metric("cv_f1_macro_std", np.std(cv_results['test_f1_macro']))
            mlflow.log_metric("cv_fit_time_mean", np.mean(cv_results['fit_time']))

            best_fold_index = np.argmax(cv_results['test_f1_macro'])
            best_model = cv_results['estimator'][best_fold_index]
            mlflow.log_metric("best_fold_index", best_fold_index)
            
            embedding_dim = X_embeddings.shape[1]
            mlflow.log_metric("actual_num_features", embedding_dim)

            y_pred = best_model.predict(X_test)

            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
            rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
            f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

            mlflow.log_metric("test_accuracy", acc)
            mlflow.log_metric("test_precision", prec)
            mlflow.log_metric("test_recall", rec)
            mlflow.log_metric("test_f1_score", f1)

            mlflow.sklearn.log_model(
                sk_model=best_model,
                name="model"
            )

    del encoder
    gc.collect()
    torch.cuda.empty_cache()
